In [1]:
import os
os.environ['KERAS_BACKEND'] = 'torch'    # has to be set before keras is imported

import numpy as np
import keras

from keras.layers import Conv2D, Resizing, Add
from custom_layers import Conv2Dx2Block, DownscaleBlock, UpscaleBlock
from model_base import ModelBase, plot_history, visualize_result
import cv2
import optuna
from matplotlib import pyplot as plt
plt.rcParams['figure.figsize'] = [15, 5]

In [2]:
# model adapted for use with optuna
class OptimizableModel(ModelBase):
    def __init__(self, filters=32, filters_last=128, kernel_size=3, kernel_size_add=2, learning_rate=0.002, gradient_accumulation_steps=4):
        self.filters = filters
        self.filters_last = filters_last
        self.kernel_size = kernel_size
        self.kernel_size_add = kernel_size_add
        super().__init__(f'opt_{filters}_{filters_last}_{kernel_size}_{learning_rate}_{gradient_accumulation_steps}', batch_size=16)

    def create(self, inputs):
        filters = [self.filters, self.filters * 2, self.filters * 4]
        x1, p = DownscaleBlock(filters[0], kernel_size=self.kernel_size+self.kernel_size_add)(inputs)
        x2, p = DownscaleBlock(filters[1], kernel_size=self.kernel_size)(p)
        
        x = Conv2Dx2Block(filters[2])(p)
        
        x = UpscaleBlock(filters[1], concat=True, kernel_size=self.kernel_size)(x, x2)
        x = UpscaleBlock(filters[0], concat=True, kernel_size=self.kernel_size+self.kernel_size_add)(x, x1)

        x = UpscaleBlock(self.filters_last)(x)

        residual = Conv2D(3, kernel_size=1, padding="same", activation=None, kernel_initializer=keras.initializers.RandomNormal(stddev=1e-3), bias_initializer="zeros")(x)
        base = Resizing(128, 128, interpolation="bicubic")(inputs)
        return Add()([base, residual])

# pruning callback for trials displaying poor validation loss
class PruningCallback(keras.callbacks.Callback):
    def __init__(self, trial):
        super().__init__()
        self.trial = trial

    def on_epoch_end(self, epoch, logs=None):
        self.trial.report(logs["val_loss"], step=epoch)
        if self.trial.should_prune():
            raise optuna.TrialPruned()

def objective(trial):
    """Objective function that controls the quality of the different runs.
    
    Args:
        trial (optuna.trial.Trial) Wrapper that controls the launching and the
            hyper-parameters of the different runs.
            
    Returns:
        (float) The quality metric of the current run.
        
    """
    # set the seed before every trial
    keras.utils.set_random_seed(1234)

    learning_rate = trial.suggest_float("learning_rate", 0.0002, 0.002, log=True)
    gradient_accumulation_steps = trial.suggest_int("gradient_accumulation_steps", 1, 4)
    model = OptimizableModel(
        filters=trial.suggest_categorical("filters", [32, 64]),
        filters_last=trial.suggest_categorical("filters_last", [32, 64, 128]),
        kernel_size=trial.suggest_categorical("kernel_size", [3, 5]),
        kernel_size_add=trial.suggest_categorical("kernel_size_add", [0, 2]),
        learning_rate=learning_rate,
        gradient_accumulation_steps=gradient_accumulation_steps
    )
    model.fit(
        epochs=20,
        learning_rate=learning_rate,
        gradient_accumulation_steps=gradient_accumulation_steps,
        pruning_callback=PruningCallback(trial)
    )
    # return model's performance on validation set
    return model.evaluate()

In [3]:
# Create Optuna study
study = optuna.create_study(direction="minimize", pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=5))

# Launch hyper-parameter search
study.optimize(objective, n_trials=20)

print('Number of finished trials:', len(study.trials))

# Show detailed info about finished trials
study.trials_dataframe()

[I 2026-11-15 23:40:55,641] A new study created in memory with name: no-name-24460534-1dad-423a-9178-3881aadcd434


Epoch 1/20
596/596 - 148s - 248ms/step - loss: 0.0231 - val_loss: 0.0215 - learning_rate: 8.7835e-04
Epoch 2/20
596/596 - 151s - 254ms/step - loss: 0.0229 - val_loss: 0.0216 - learning_rate: 8.7835e-04
Epoch 3/20
596/596 - 148s - 248ms/step - loss: 0.0229 - val_loss: 0.0215 - learning_rate: 8.7835e-04
Epoch 4/20
596/596 - 145s - 243ms/step - loss: 0.0229 - val_loss: 0.0215 - learning_rate: 8.7835e-04
Epoch 5/20
596/596 - 138s - 232ms/step - loss: 0.0228 - val_loss: 0.0215 - learning_rate: 8.7835e-04
Epoch 6/20
596/596 - 141s - 236ms/step - loss: 0.0229 - val_loss: 0.0214 - learning_rate: 8.7835e-04
Epoch 7/20
596/596 - 144s - 242ms/step - loss: 0.0224 - val_loss: 0.0215 - learning_rate: 8.7835e-04
Epoch 8/20
596/596 - 137s - 230ms/step - loss: 0.0216 - val_loss: 0.0197 - learning_rate: 8.7835e-04
Epoch 9/20
596/596 - 150s - 252ms/step - loss: 0.0211 - val_loss: 0.0192 - learning_rate: 8.7835e-04
Epoch 10/20
596/596 - 146s - 245ms/step - loss: 0.0208 - val_loss: 0.0191 - learning_rate: 

[W1116 00:12:50.750838223 CUDACachingAllocator.cpp:3934] memory allocation failed with OOM on device 0 while trying to allocate 876609536 bytes (free: 407699456, total: 8154513408).
[W1116 00:12:50.777540854 CUDACachingAllocator.cpp:3934] memory allocation failed with OOM on device 0 while trying to allocate 876609536 bytes (free: 428670976, total: 8154513408).
[W1116 00:12:50.800317031 CUDACachingAllocator.cpp:3934] memory allocation failed with OOM on device 0 while trying to allocate 876609536 bytes (free: 428670976, total: 8154513408).
[W1116 00:12:50.800415801 CUDACachingAllocator.cpp:3934] memory allocation failed with OOM on device 0 while trying to allocate 876609536 bytes (free: 428670976, total: 8154513408).


596/596 - 171s - 286ms/step - loss: 0.0205 - val_loss: 0.0188 - learning_rate: 8.7835e-04
Epoch 14/20
596/596 - 189s - 318ms/step - loss: 0.0205 - val_loss: 0.0187 - learning_rate: 8.7835e-04
Epoch 15/20
596/596 - 179s - 300ms/step - loss: 0.0204 - val_loss: 0.0189 - learning_rate: 8.7835e-04
Epoch 16/20
596/596 - 188s - 316ms/step - loss: 0.0204 - val_loss: 0.0188 - learning_rate: 8.7835e-04
Epoch 17/20
596/596 - 173s - 291ms/step - loss: 0.0203 - val_loss: 0.0188 - learning_rate: 8.7835e-04
Epoch 18/20
596/596 - 180s - 302ms/step - loss: 0.0203 - val_loss: 0.0187 - learning_rate: 8.7835e-04
Epoch 19/20
596/596 - 175s - 293ms/step - loss: 0.0203 - val_loss: 0.0186 - learning_rate: 8.7835e-04
Epoch 20/20
596/596 - 176s - 295ms/step - loss: 0.0202 - val_loss: 0.0187 - learning_rate: 8.7835e-04


[I 2026-11-16 00:35:31,455] Trial 0 finished with value: 0.01981283538043499 and parameters: {'learning_rate': 0.000878352256999843, 'gradient_accumulation_steps': 3, 'filters': 32, 'filters_last': 128, 'kernel_size': 5, 'kernel_size_add': 0}. Best is trial 0 with value: 0.01981283538043499.
[W 2026-11-16 00:35:31,602] Trial 1 failed with parameters: {'learning_rate': 0.0007037277324830806, 'gradient_accumulation_steps': 1, 'filters': 32, 'filters_last': 128, 'kernel_size': 5, 'kernel_size_add': 2} because of the following error: ValueError('`gradient_accumulation_steps` must be an integer >= 2. Received: gradient_accumulation_steps=1').
Traceback (most recent call last):
  File "/home/tto/workspace/venvs/cv/lib/python3.12/site-packages/optuna/study/_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "/tmp/ipykernel_117224/2567630199.py", line 59, in objective
    model.fit(
  File "/home/tto/workspace/cv-course-project/mode

ValueError: `gradient_accumulation_steps` must be an integer >= 2. Received: gradient_accumulation_steps=1

In [ ]:
metrics = [trial.value for trial in study.trials]
plt.plot(metrics, '.-'), plt.grid(True)
plt.xlabel('trial'), plt.ylabel('metric')

In [ ]:
# Show best trial
trial = study.best_trial
print('Best trial:', trial.number, '\n')
print('  Metric:', trial.value)
print('  Params:')
for key, value in trial.params.items():
    print('\t', key.ljust(13), ':', value)

In [ ]:
# Show overall stats
mu = np.mean([trial.value for trial in study.trials])
std = np.std([trial.value for trial in study.trials])

print('Avg metric:', np.mean(mu))
print('Std metric:', np.mean(std))

In [ ]:
importances = optuna.importance.get_param_importances(study)
plt.barh(list(importances)[::-1], list(importances.values())[::-1])
plt.xlabel('Importance'), plt.grid(True)

In [ ]:
for t in study.trials:
    print(t)